# LayerQuantizer Demo: WeatherBench 2 Compression

This notebook demonstrates **LayerQuantizer**, a compressor designed for gridded geophysical data. We will benchmark its performance against standard Zarr/Xarray compression (Blosc/Zstd) using WeatherBench 2 (ERA5) data.

This demo covers:
1.  **Zarr 2** Benchmarks (Compression Ratio, Speed, Accuracy)
2.  **Zarr 3** Benchmarks (Speed, Accuracy)


## 1. Installation and Setup


In [1]:
# Install LayerQuantizer and dependencies
# Using git+https to install directly from the repository
%pip install -q git+https://github.com/csubich/layerquantizer.git@dev tqdm xarray zarr gcsfs tabulate


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import xarray as xr
import zarr
import numpy as np
import pandas as pd
import time
import fsspec
from layerquantizer import LayerQuantizer
from numcodecs import Blosc
from tabulate import tabulate
import warnings
from tqdm.auto import tqdm
from pathlib import Path

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning)


## 2. Utilities


In [3]:
# Constants
# We use the coarse dataset for faster demonstration in Colab
URL_COARSE = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr"
URL_FINE = "gs://weatherbench2/datasets/era5/1959-2023_01_10-full_37-1h-0p25deg-chunk-1.zarr"

BENCHMARK_RESOLUTION="fine"

VARIABLES = [
    "geopotential",
    "potential_vorticity",
    "specific_humidity",
    "temperature",
    "u_component_of_wind",
    "v_component_of_wind",
    "vertical_velocity",
]

# Cache Directory
CACHE_DIR = Path("data/cache")

def get_wb_era5(resolution="fine", chunks=None, cache=True):
    # Use token='anon' for public access to WeatherBench2
    if (resolution == 'coarse'):
      url = URL_COARSE
    elif (resolution == 'fine'):
      url = URL_FINE
    else:
      raise ValueError(f'Unkown {resolution=}')

    storage_options = {"token": "anon"}

    if cache:
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        # Use CachingFileSystem (simplecache)
        protocol = url.split("://")[0]
        path = url.split("://")[-1]
        fs = fsspec.filesystem(
            "simplecache",
            target_protocol=protocol,
            target_options=storage_options,
            cache_storage=str(CACHE_DIR)
        )
        mapper = fs.get_mapper(path)
        ds = xr.open_zarr(mapper, chunks=chunks)
    else:
        ds = xr.open_zarr(url, chunks=chunks, storage_options=storage_options)
    return ds

def sample_data(ds, n_time=5, seed=42):
    # Select random time slices
    gen = np.random.default_rng(seed=seed)
    sel_times = np.sort(gen.choice(ds.time.data, size=n_time, replace=False))
    return ds.sel(time=sel_times)

def get_store_size(store):
    # Calculate size of store in bytes
    if hasattr(store, 'keys'): # Zarr 2
        return sum(store.getsize(k) for k in store.keys())
    # Zarr 3 (approximate for MemoryStore)
    if hasattr(store, '_store_dict'):
         return sum(len(v) for v in store._store_dict.values())
    if hasattr(store, '_store'):
         return sum(len(v) for v in store._store.values())
    return 0

def calculate_stats(original, reconstructed, compressed_size, duration, mode="write"):
    # Metrics
    original_nbytes = original.nbytes
    ratio = original_nbytes / compressed_size if compressed_size > 0 else 0
    speed = (original_nbytes / (1024**2)) / duration if duration > 0 else 0

    # Full-Scale Error
    max_val = original.max().values
    min_val = original.min().values
    fs_range = max_val - min_val
    if fs_range == 0:
        fs_err = 0.0
    else:
        diff = np.abs(original - reconstructed)
        fs_err = (diff / fs_range).max().values

    return {
        "Ratio": ratio,
        "FS Error": fs_err,
        f"{mode.capitalize()} (MiB/s)": speed,
        "Compressed (MiB)": compressed_size / (1024**2)
    }


## 3. Zarr 2 Benchmark

We compare:
1.  **Default**: `Blosc(cname='zstd', clevel=5)`
2.  **LayerQuantizer**: `LayerQuantizer(nbits=16, clevel=[1, 5, 9])`


In [4]:
# Load Data
ds = get_wb_era5(BENCHMARK_RESOLUTION)
ds_subset = sample_data(ds, n_time=5)

In [5]:
results_z2 = []

# Configurations
configs = [
    {'name': 'Default (LZ4)', 'compressor': None},
    {'name': 'LQ (16, LZ4)', 'compressor': LayerQuantizer(nbits=16, blosc_cname='lz4', blosc_clevel=1)},
    {'name': 'Zstd 5', 'compressor' : Blosc(cname='zstd', clevel=5)},
    {'name': 'LQ (16, Zstd 1)', 'compressor': LayerQuantizer(nbits=16, blosc_cname='zstd', blosc_clevel=1)},
    {'name': 'LQ (16, Zstd 5)', 'compressor': LayerQuantizer(nbits=16, blosc_cname='zstd', blosc_clevel=5)},
    {'name': 'LQ (16, Zstd 9)', 'compressor': LayerQuantizer(nbits=16, blosc_cname='zstd', blosc_clevel=9)},
]

print("Running Zarr 2 Benchmark...")
pbar = tqdm(total=len(VARIABLES) * len(configs))
for var_name in VARIABLES:
    da = ds_subset[var_name].compute() # Load into memory
    da = da.drop_encoding()

    for cfg in configs:
        pbar.set_description(f"Zarr 2: {var_name} - {cfg['name']}")
        # Write
        store = zarr.storage.MemoryStore()
        start = time.time()
        encoding = {}
        if cfg['compressor'] is not None:
           encoding = {var_name: {'compressor': cfg['compressor']}}
        da.to_zarr(store, mode='w', zarr_format=2, encoding=encoding, compute=True)
        write_time = time.time() - start

        # Size
        comp_size = get_store_size(store)

        # Read
        start = time.time()
        da_rec = xr.open_zarr(store)[var_name].load()
        read_time = time.time() - start

        stats = calculate_stats(da, da_rec, comp_size, write_time, "write")
        read_stats = calculate_stats(da, da_rec, comp_size, read_time, "read")

        res = {
            "Variable": var_name,
            "Config": cfg['name'],
            "Ratio": stats['Ratio'],
            "FS Error": stats['FS Error'],
            "Write (MiB/s)": stats['Write (MiB/s)'],
            "Read (MiB/s)": read_stats['Read (MiB/s)'],
        }
        results_z2.append(res)
        pbar.update(1)

pbar.close()
print("Done.")


Running Zarr 2 Benchmark...


  0%|          | 0/42 [00:00<?, ?it/s]

Done.


## 4. Zarr 3 Benchmark

Zarr 3 uses a new codec pipeline approach. LayerQuantizer works as an array-to-bytes codec (serializer) followed by a bytes-to-bytes codec (compressor).

*Note: Compression ratios are omitted here to keep the demo concise, focusing on performance parity and error.*  In like-for-like configurations the compression ratios will be nearly identical between Zarr 2 and 3, with differences due to the Zarr metadata.


In [6]:
try:
    from layerquantizer import LayerQuantizerCodec
    from zarr.codecs import BloscCodec, ZstdCodec

    results_z3 = []

    # Zarr 3 Configs (Chained)
    configs_z3 = [
        {
            'name': 'Default (Zstd 0)',
            'encoding_kwargs': {}
        },
        {
            'name': 'Zstd 5',
            'encoding_kwargs': {
                'compressors': [ZstdCodec(level=5)]
            }
        },
        {
            'name': 'Blosc+Zstd 5',
            'encoding_kwargs': {
                'compressors': [BloscCodec(cname='zstd', clevel=5, shuffle='shuffle')]
            }
        },
        {
            'name': 'LQ (16) + Blosc + Zstd 1',
            'encoding_kwargs': {
                'serializer': LayerQuantizerCodec(nbits=16),
                'compressors': [BloscCodec(cname='zstd', clevel=1, shuffle='shuffle')]
            }
        },
        {
            'name': 'LQ (16) + Blosc + Zstd 5',
            'encoding_kwargs': {
                'serializer': LayerQuantizerCodec(nbits=16),
                'compressors': [BloscCodec(cname='zstd', clevel=5, shuffle='shuffle')]
            }
        },
    ]

    print("Running Zarr 3 Benchmark...")
    pbar = tqdm(total=len(VARIABLES) * len(configs_z3))

    for var_name in VARIABLES:
        da = ds_subset[var_name].compute()
        da = da.drop_encoding()

        for cfg in configs_z3:
            pbar.set_description(f"Zarr 3: {var_name} - {cfg['name']}")

            store = zarr.storage.MemoryStore()

            # Map the config to the xarray encoding dictionary
            encoding = {var_name: cfg['encoding_kwargs']} if cfg['encoding_kwargs'] else {}

            # Write
            start = time.time()
            da.to_zarr(store, mode='w', zarr_format=3, encoding=encoding, compute=True)
            write_time = time.time() - start

            # Read back
            start = time.time()
            # consolidated=False suppresses warnings for some Zarr 3 versions
            da_rec = xr.open_zarr(store, consolidated=False)[var_name].load()
            read_time = time.time() - start

            # Size
            comp_size = get_store_size(store)

            # Calculate Stats
            stats = calculate_stats(da, da_rec, comp_size, write_time, "write")
            read_stats = calculate_stats(da, da_rec, comp_size, read_time, "read")

            results_z3.append({
                "Variable": var_name,
                "Config": cfg['name'],
                "FS Error": stats['FS Error'],
                "Ratio": stats['Ratio'],
                "Write (MiB/s)": stats['Write (MiB/s)'],
                "Read (MiB/s)": read_stats['Read (MiB/s)'],
            })

            pbar.update(1)

    pbar.close()
    print("Done.")

except ImportError:
    print("Zarr 3 or LayerQuantizerCodec not available/compatible in this environment.")
except Exception as e:
    print(f"Zarr 3 Benchmark failed: {e}")

Running Zarr 3 Benchmark...


  0%|          | 0/35 [00:00<?, ?it/s]

Done.


## 5. Results

Summary of Compression Ratios, Performance, and Errors.


In [7]:
def pivot_table(data, value_col, agg_func='mean',floatfmt=".2f"):
    df = pd.DataFrame(data)
    if df.empty:
        return
    # 1. Capture the order of appearance
    config_order = df['Config'].unique()
    pivot = df.pivot_table(index='Variable', columns='Config', values=value_col, aggfunc=agg_func)
    # 2. Reindex columns to match appearance order
    pivot = pivot.reindex(columns=config_order)
    print(tabulate(pivot, headers='keys', tablefmt='github', floatfmt=floatfmt))

print("### Zarr 2: Compression Ratio (Higher is Better)")
pivot_table(results_z2, 'Ratio')

print("\n### Zarr 3: Compression Ratio (Higher is Better)")
pivot_table(results_z3, 'Ratio')

print(f"\n### Zarr 2: Full-Scale Error (Lower is Better), Reference {2**-17:.2e}")
# Format error as scientific notation
pivot_table(results_z2, 'FS Error',floatfmt=".2e")

print("\n### Zarr 2: Write Speed (MiB/s)")
pivot_table(results_z2, 'Write (MiB/s)')

print("\n### Zarr 3: Write Speed (MiB/s)")
pivot_table(results_z3, 'Write (MiB/s)')

print("\n### Zarr 2: Read Speed (MiB/s)")
pivot_table(results_z2, 'Read (MiB/s)')

print("\n### Zarr 3: Read Speed (MiB/s)")
pivot_table(results_z3, 'Read (MiB/s)')


print("\n### Zarr 2: Performance Comparison (Average MiB/s)")
df_z2 = pd.DataFrame(results_z2)
config_order = df_z2['Config'].unique()
speed_summary = df_z2.groupby('Config')[['Write (MiB/s)', 'Read (MiB/s)']].mean()
speed_summary = speed_summary.reindex(config_order)
print(tabulate(speed_summary, headers='keys', tablefmt='github', floatfmt=".1f"))

print("\n### Zarr 3: Performance Comparison (Average MiB/s)")
df_z3 = pd.DataFrame(results_z3)

config_order = df_z3['Config'].unique()
speed_summary = df_z3.groupby('Config')[['Write (MiB/s)', 'Read (MiB/s)']].mean()
speed_summary = speed_summary.reindex(config_order)
print(tabulate(speed_summary, headers='keys', tablefmt='github', floatfmt=".1f"))


### Zarr 2: Compression Ratio (Higher is Better)
| Variable            |   Default (LZ4) |   LQ (16, LZ4) |   Zstd 5 |   LQ (16, Zstd 1) |   LQ (16, Zstd 5) |   LQ (16, Zstd 9) |
|---------------------|-----------------|----------------|----------|-------------------|-------------------|-------------------|
| geopotential        |            1.90 |           3.81 |     2.22 |              5.46 |              5.51 |              5.70 |
| potential_vorticity |            1.40 |           2.81 |     1.62 |              3.62 |              3.72 |              3.95 |
| specific_humidity   |            1.47 |           2.85 |     1.67 |              3.53 |              3.62 |              3.78 |
| temperature         |            1.80 |           3.16 |     1.98 |              3.74 |              3.85 |              4.07 |
| u_component_of_wind |            1.30 |           3.12 |     1.42 |              3.67 |              3.78 |              4.03 |
| v_component_of_wind |            1.26 |